In [8]:
import os
import sys
import pickle
from obspy.taup import taup_create

import dask
from dask.distributed import Client as dask_Client
from dask.distributed import LocalCluster as dask_LocalCluster

path = '../code'
sys.path.insert(0, path)

import SeisScan as ss

# Creake DASK workers for parallel processing

In [2]:
# Create dask workers

os.cpu_count()

n_workers = os.cpu_count() - 2

dask_cluster = dask_LocalCluster()
dask_client = dask_Client(dask_cluster)
dask_cluster.scale(n_workers)

# Earth model

In [ ]:
model_name = 'okl.tvel'
taup_create.build_taup_model(model_name, verbose=False)

# Prepare P- and S-wave travel time lookup table

In [7]:
print('Preparation of travel time lookup table')

min_dist_k, max_dist_k, step_dist_k = 0.0, 50.0, 2.0    # start, end and step epi. distance in km
min_dep_k, max_dep_k, step_dep_k = 0.0, 15.0, 0.5       # start, end and step depth in km
model = 'okl'

phase = 'p'
mod_dist_r1, mod_dep_r1, mod_ttp_r2, computation_time = ss.prepare_traveltime_lookup_table(
    min_dist_k, max_dist_k, step_dist_k,
    min_dep_k, max_dep_k, step_dep_k,
    phase, model_name=model, dask_client=dask_client
)
print(f'{phase}-wave travel time computation takes {computation_time} sec')


phase = 's'
_, _, mod_tts_r2, computation_time = ss.prepare_traveltime_lookup_table(
    min_dist_k, max_dist_k, step_dist_k,
    min_dep_k, max_dep_k, step_dep_k,
    phase, model_name=model, dask_client=dask_client
)
print(f'{phase}-wave travel time computation takes {computation_time} sec')

print('----------------------------------------------------')

Preparation of travel time lookup table
p-wave travel time computation takes 4.761185169219971 sec
s-wave travel time computation takes 6.585022926330566 sec
----------------------------------------------------


# Save the travel time

In [12]:
tt_lookup_table = {
    'mod_dist_r1':mod_dist_r1,
    'mod_dep_r1':mod_dep_r1,
    'mod_ttp_r2':mod_ttp_r2,
    'mod_tts_r2':mod_tts_r2,
}

pickle.dump(tt_lookup_table, open('tt_lookup_table.p', 'wb'))

# Close all DASK workers

In [14]:
#--- close dask client
dask_client.close()
dask_cluster.close()